In [1]:
import pandas as pd
import xarray as xr
import numpy as np

In [2]:
def load_region_matrix(csv_path, lat_size=360, lon_size=720):
    df = pd.read_csv(csv_path)

    region_matrix = np.empty((lat_size, lon_size), dtype=object)
    region_matrix[:] = None  # 默认填 None

    for _, r in df.iterrows():
        I = int(r["I"]) - 1
        J = int(r["J"]) - 1
        region_matrix[I, J] = r["Rall"]

    return region_matrix


In [3]:
def split_variable_by_region(da, region_matrix):

    time = da["time"]
    lat  = da["lat"]
    lon  = da["lon"]

    # ---- 强健、不会报错的地域收集方式 ----
    regions = set()
    for r in region_matrix.flatten():
        if isinstance(r, str) and r.strip() != "":
            regions.add(r)
    regions = sorted(regions)

    # ---- 转 region_matrix 为 xarray ----
    region_da = xr.DataArray(
        region_matrix,
        coords={"lat": lat, "lon": lon},
        dims=("lat", "lon")
    )

    out = {}

    for reg in regions:
        mask = (region_da == reg)

        # 使用 xarray.where → 内存安全，不复制大数组
        da_reg = da.where(mask)
        da_reg.name = f"{reg}_{da.name}"

        out[reg] = da_reg

    return out

In [4]:
def write_region_split_nc(output_path, region_dict):
    ds = xr.Dataset()
    for reg, da in region_dict.items():
        varname = da.name.replace(" ", "_")
        ds[varname] = da
    ds.to_netcdf(output_path)
    print(f"[Saved] {output_path}")

In [5]:
def run_full_split(csv_path, nc_path, output_prefix):
    region_matrix = load_region_matrix(csv_path)
    ds = xr.open_dataset(nc_path)

    variable_groups = [
        ("region_agri",       "region", "agri"),
        ("region_forest",     "region", "forest"),
        ("region_grassland",  "region", "grassland"),
        ("basin_agri",        "basin",  "agri"),
        ("basin_forest",      "basin",  "forest"),
        ("basin_grassland",   "basin",  "grassland")
    ]

    for varname, mode, landtype in variable_groups:
        print(f"\n========== Processing {varname} ==========")

        da = ds[varname]   # shape = (time,lat,lon)
        da.name = f"{landtype}"

        region_dict = split_variable_by_region(da, region_matrix)

        output_file = f"{mode}_{landtype}_17regions.nc"
        write_region_split_nc(output_file, region_dict)

In [6]:
CSV_PATH = "../../CSV/RIJ.csv"
NC_PATH  = "../../NC/compare.nc"
OUTPUT_PREFIX = "split"

run_full_split(CSV_PATH, NC_PATH, OUTPUT_PREFIX)


========== Processing region_agri ==========


MemoryError: Unable to allocate 10.9 MiB for an array with shape (11, 360, 720) and data type float32